<a href="https://colab.research.google.com/github/takatakamanbou/AdvML/blob/2025/AdvML2025_ex15notebookA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AdvML ex15notebookA

<img width=72 src="https://www-tlab.math.ryukoku.ac.jp/~takataka/course/AdvML/AdvML-logo.png"> [この授業のウェブページ](https://www-tlab.math.ryukoku.ac.jp/wiki/?AdvML)




板書や口頭で補足する前提なので，この notebook だけでは説明が不完全です．


---
## 画像の生成
---

近年の画像生成の手法は，これまで紹介した深層生成モデルを拡張・発展したものとなっている．

---
### 条件付き生成

生成モデルは，データ $\pmb{x}$ の分布 $p(\pmb{x})$ を学習し，それをもとに新たなデータを生み出すモデルである．しかし，このままでは，生成されるデータの内容を制御することができない．例えば，0から9までの10クラスの手書き数字データで生成モデルを学習させたとしても，「3の画像が欲しい」というように指定して生成させることはできない．

これに対して **条件付き生成モデル** (**conditional generative model**) では，データに対応する条件 $\pmb{y}$ （例: クラスラベルや属性）を使って，条件付き分布 $p(\pmb{x}|\pmb{y})$ を学習する．これにより，特定の条件に合ったデータを生成することが可能になる．通常は，モデルの入力に $\pmb{y}$ も追加する形で実現される．

---
### 実験: Conditional VAE による手書き数字画像の生成

VAE（変分オートエンコーダ）を条件付きに拡張した Conditional VAE に手書き数字画像を学習させてみよう．

ここでは，VAE のエンコーダ/デコーダのそれぞれへの入力に，10通りのクラスラベルの情報も追加する形でモデルを作る．クラスラベルの情報は one-hot エンコーディング（注）して 10 次元ベクトルとし，エンコーダでは画素値と連結して入力，デコーダでは潜在変数の値と連結して入力する．

<br>
<hr width="50%" align="left">
<span style="font-size: 75%">
※注: ベクトルの要素のうち 1 つだけが 1 で他は 0 になるように符号化すること．
</span>

#### いろいろ import

In [ ]:
# 準備あれこれ
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn
seaborn.set_theme()

# scikit-learn のいろいろ
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

# NumPy の 疑似乱数生成器（rng = random number generator）
from numpy.random import default_rng
rng = default_rng() # 疑似乱数生成器を初期化

# PyTorch 関係のほげ
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import ToTensor

!pip install torchinfo
from torchinfo import summary

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

#### データの準備

In [ ]:
# MNIST データセットの入手
Xraw, yraw = fetch_openml('mnist_784', version=1, parser='auto', return_X_y=True, as_frame=False)
Xall = Xraw[:20000] / 255.0     # 画素値が [0, 255] の整数値なので [0, 1] の浮動小数点数値に変換
yall = yraw[:20000].astype(int) # クラスラベル．0 から 9 の整数値

# 学習データとテストデータの分割
XL, XT, yL, yT = train_test_split(Xall, yall, test_size=4000, random_state=4649, stratify=yall)
print(XL.shape, yL.shape)
print(XT.shape, yT.shape)
NL, D = XL.shape
NT = len(XT)

K = 10

# 平均を引いたデータを用意
Xm = np.mean(XL, axis=0)
XL2 = XL - Xm
XT2 = XT - Xm

In [ ]:
# 学習データの最初の50枚を可視化
nrow, ncol = 5, 10
fig, ax = plt.subplots(nrow, ncol, figsize=(0.6*ncol, 0.6*nrow))
for i in range(nrow):
    for j in range(ncol):
        img = XL[i*ncol + j, ::].reshape((28, 28))
        ax[i, j].imshow(img, cmap=plt.cm.gray, vmin=0, vmax=1)
        ax[i, j].axis('off')

fig.tight_layout()
plt.show()

#### データを扱うクラスの定義

In [ ]:
# データを扱うためのクラス
#
class MMDataset2(Dataset):

    def __init__(self, dataX, lab):
        self.X = dataX
        self.lab = lab

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        X = torch.tensor(self.X[idx], dtype=torch.float32)
        y = torch.tensor(self.lab[idx], dtype=torch.long)
        return X, y

#### Conditional VAE モデルの定義

「XとYを連結して入力」「ZとYを連結して入力」とコメントを付した部分以外は，VAEそのまま．

In [ ]:
### CVAE Encoder
#
class CVAEEncoder(nn.Module):

    def __init__(self, dimX, dimY, dimHidden, dimZ):
        super().__init__()
        self.layer1       = nn.Linear(dimX + dimY, dimHidden) # X と Y を連結して入力
        self.layer2_mu    = nn.Linear(dimHidden, dimZ)
        self.layer2_logvar = nn.Linear(dimHidden, dimZ)

    def forward(self, X, Y):
        XY = torch.cat([X, Y], dim=1) # X と Y を連結して入力
        H = F.relu(self.layer1(XY))
        mu     = self.layer2_mu(H)
        logvar = self.layer2_logvar(H)
        return mu, logvar


### CVAE Decoder
#
class CVAEDecoder(nn.Module):

    def __init__(self, dimZ, dimY, dimHidden, dimXt):
        super().__init__()
        self.layer1 = nn.Linear(dimZ + dimY, dimHidden) # Z と Y を連結して入力
        self.layer2 = nn.Linear(dimHidden, dimXt)

    def forward(self, Z, Y):
        ZY = torch.cat([Z, Y], dim=1) # Z と Y を連結して入力
        H = F.relu(self.layer1(ZY))
        Xt = self.layer2(H)
        return Xt

### reparameterization trick
#
def reparameterization(mu, logvar):
    sigma = torch.exp(0.5*logvar)
    eps = torch.randn_like(sigma)
    Z = mu + sigma * eps
    return Z

### Conditional VAE
#
class ConditionalVAE(nn.Module):

    def __init__(self, dimX, dimY, dimHidden, dimZ):
        super().__init__()
        self.encoder = CVAEEncoder(dimX, dimY, dimHidden, dimZ)
        self.decoder = CVAEDecoder(dimZ, dimY, dimHidden, dimX)

    def forward(self, X, Y):
        mu, logvar = self.encoder(X, Y)
        Z = reparameterization(mu, logvar)
        Xt = self.decoder(Z, Y)
        return Xt, mu, logvar

    def reconstruct(self, X, Y):
        mu, logvar = self.encoder(X, Y)
        Xt = self.decoder(mu, Y)
        return Xt

    def loss(self, Xt, X, mu, logvar):
        SQE = F.mse_loss(Xt, X, reduction='sum')
        KLD = - torch.sum(1 + logvar - mu**2 - logvar.exp())
        return SQE, KLD

#### 学習と評価の関数の定義

In [ ]:
# 学習の関数
#
def trainCVAE(model, optimizer, dl):
    loss_sum = sqe_sum = kld_sum = 0.0
    n = 0
    for i, (X, lab) in enumerate(dl):
        # 一つのバッチをモデルに入力して出力を得る
        X = X.to(device)
        Y = F.one_hot(lab, num_classes=10).float().to(device)
        Xt, mu, logvar = model(X, Y)
        # 損失関数の値を計算
        sqe, kld = model.loss(Xt, X, mu, logvar)
        loss = sqe + kld
        n += len(X)
        loss_sum += loss.item()
        sqe_sum += sqe.item()
        kld_sum += kld.item()
        # 誤差逆伝播学習
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return loss_sum/n, sqe_sum/n, kld_sum/n


# 損失関数の値を求める関数
#
@torch.no_grad()
def evaluateCVAE(model, dl):
    loss_sum = sqe_sum = kld_sum = 0.0
    n = 0
    for i, (X, lab) in enumerate(dl):
        # 一つのバッチをモデルに入力して出力を得る
        X = X.to(device)
        Y = F.one_hot(lab, num_classes=10).float().to(device)
        Xt, mu, logvar = model(X, Y)  # 一つのバッチ X, Y を入力して出力を計算
        # 損失関数の値を計算
        sqe, kld = model.loss(Xt, X, mu, logvar)
        loss = sqe + kld
        n += len(X)
        loss_sum += loss.item()
        sqe_sum += sqe.item()
        kld_sum += kld.item()

    return loss_sum/n, sqe_sum/n, kld_sum/n

#### 学習

条件付きモデルになったこと以外，ハイパーパラメータは ex13notebookA の実験と同じ．

In [ ]:
# ハイパーパラメータ
dimHidden = 1000 # 隠れ層ニューロン数
dimZ = 100       # 潜在変数の次元数
nepoch = 100     # 学習エポック数
bsize = 100      # バッチサイズ
lr = 1e-3        # 学習率

# データ読み込みの仕組みを作る
dsL = MMDataset2(XL2, yL)
dsT = MMDataset2(XT2, yT)
dlL = DataLoader(dsL, batch_size=bsize, shuffle=True)
dlT = DataLoader(dsT, batch_size=bsize, shuffle=False)

# ネットワークモデルの定義
cvae = ConditionalVAE(D, K, dimHidden, dimZ).to(device)

# ネットワークの構造を表示
print(cvae)
print(summary(cvae, input_size=[(1, D), (1, K)]))

# パラメータ最適化器の設定
optimizer = torch.optim.Adam(cvae.parameters(), lr=lr)

# 学習
L = []
print(f'学習データ数: {len(dsL)}  テストデータ数: {len(dsT)}')
print()
print('# epoch  lossL  sqeL  kldL  lossT  sqeT  kldT')
for t in range(1, nepoch+1):
    lossL, sqeL, kldL = trainCVAE(cvae, optimizer, dlL)
    lossL, sqeL, kldL = lossL/D, sqeL/D, kldL/D
    lossT, sqeT, kldT = evaluateCVAE(cvae, dlT)
    lossT, sqeT, kldT = lossT/D, sqeT/D, kldT/D
    L.append([t, lossL, lossT])
    if (t < 10) or (t % 10 == 0):
        print(f'{t}   {lossL:.5f}  {sqeL:.5f}  {kldL:.5f}   {lossT:.5f}  {sqeT:.5f}  {kldT:.5f}')

# 学習曲線の表示
data = np.array(L)
fig, ax = plt.subplots(1, 1)
ax.plot(data[:, 0], data[:, 1], '.-', label='loss for training data')
ax.plot(data[:, 0], data[:, 2], '.-', label='loss for test data')
ax.axhline(0.0, color='gray')
ax.legend()
ax.set_title(f'loss')
plt.show()

# 学習後の損失と識別率
lossL, sqeL, kldL = evaluateCVAE(cvae, dlL)
print(f'# lossL: {lossL/D:.5f}', end='   ')
lossT, sqeT, kldT = evaluateCVAE(cvae, dlT)
print(f'# lossT: {lossT/D:.5f}')

#### 再構成と生成

In [ ]:
# テストデータ1バッチ分の再構成
for i, (X, lab) in enumerate(dlT):
    X = X.to(device)
    Y = F.one_hot(lab, num_classes=10).float().to(device)
    Xt1, mu, logvar = cvae(X, Y)
    Xt2, mu, logvar = cvae(X, Y)
    Xt3 = cvae.reconstruct(X, Y)
    break
XX     = X.to('cpu').detach().numpy() + Xm
XXrec1 = Xt1.to('cpu').detach().numpy() + Xm
XXrec2 = Xt2.to('cpu').detach().numpy() + Xm
XXrec3 = Xt3.to('cpu').detach().numpy() + Xm

# 再構成したテストデータの最初の10枚を可視化
ncol = 10
fig, ax = plt.subplots(4, ncol, figsize=(0.8*ncol, 0.8*4))

# 元画像
for j in range(ncol):
    img = XX[j, ::].reshape((28, 28))
    ax[0, j].imshow(img, cmap=plt.cm.gray, vmin=0, vmax=1)
    ax[0, j].axis('off')

# 再構成した画像（正規分布からサンプリング）
for j in range(ncol):
    img = XXrec1[j, ::].reshape((28, 28))
    ax[1, j].imshow(img, cmap=plt.cm.gray, vmin=0, vmax=1)
    ax[1, j].axis('off')

# 再構成した画像（正規分布からサンプリング）
for j in range(ncol):
    img = XXrec2[j, ::].reshape((28, 28))
    ax[2, j].imshow(img, cmap=plt.cm.gray, vmin=0, vmax=1)
    ax[2, j].axis('off')

# 再構成した画像（平均を使う）
for j in range(ncol):
    img = XXrec3[j, ::].reshape((28, 28))
    ax[3, j].imshow(img, cmap=plt.cm.gray, vmin=0, vmax=1)
    ax[3, j].axis('off')

fig.tight_layout()
plt.show()

# 再構成誤差
mse = np.mean((XX[:ncol] - XXrec3[:ncol])**2)
print(f'MSE = {mse:.5f}')

図の上から1行目がモデルへの入力 $\pmb{x}$ であり，2行目以降は3通りの再構成である．2行目と3行目はランダムサンプリングした2通りの $\pmb{z}$ を用いた再構成，4行目は，エンコーダが出力した平均の値をそのまま $\pmb{z}$ として得られた再構成である．

In [ ]:
# 潜在変数 Z の値を正規乱数でサンプリング
Zgen = np.random.normal(size=50*dimZ).reshape((50, dimZ))
Zgen = torch.tensor(Zgen.astype(np.float32)).to(device)
# 条件 Y は 0,1,2,...,9 の繰り返しに設定
Ygen = np.arange(0, 50, dtype=int) % 10
Ygen_onehot = np.eye(10)[Ygen]  # ワンホットエンコード
Ygen_onehot = torch.tensor(Ygen_onehot.astype(np.float32)).to(device)

# デコーダで生成
XXrec = cvae.decoder(Zgen, Ygen_onehot)
XXrec = XXrec.cpu().detach().numpy() + Xm

# 生成した画像50枚を可視化
nrow, ncol = 5, 10
fig, ax = plt.subplots(nrow, ncol, figsize=(0.6*ncol, 0.6*nrow))
for i in range(nrow):
    for j in range(ncol):
        img = XXrec[i*ncol + j, ::].reshape((28, 28))
        ax[i, j].imshow(img, cmap=plt.cm.gray, vmin=0, vmax=1)
        ax[i, j].axis('off')

fig.tight_layout()
plt.show()

こちらは，$\pmb{y}$ を指定して画像を生成させた結果を示す．

---
### 自然言語による条件付けと CLIP

条件付き生成モデルでは，最初はクラスラベルや属性といった構造化された情報が条件として用いられていたが，近年では，自然言語による柔軟な条件指定が可能な手法も登場している．例えば，画像を $\pmb{x}$，その内容を説明する文を符号化したものを $\pmb{y}$ とすると，条件付き分布 $p(\pmb{x}|\pmb{y})$ を適切に学習することで，「笑っている猫の写真」など自然言語の指示に従った画像生成が可能になる．

このような目的には，画像とその内容を表す自然言語テキストのペアを用いて学習された，**CLIP**（Contrastive Language–Image Pretraining） と呼ばれるモデルがよく用いられる．以下では，このモデルの概要を紹介する．

**CLIP**（Contrastive Language–Image Pretraining）は，OpenAI によって提案された，画像と言語を共通の埋め込み空間に写像することを目的としたモデルである(Radford et al., 2021)．
このモデルは，インターネット上から収集された画像と言語のペア（例：「猫が魚をくわえて走っている」というキャプション付きの画像）を用いて，**対照学習** (Contrastive Learning) という学習手法によって学習される．

CLIP は，大きく分けて次の二つのニューラルネットワークから構成される：

- Image Encoder: 画像を特徴ベクトルに変換する．ResNet や Vision Transformer（ViT）などが用いられる（注1）．
- Text Encoder: 文をトークン化し，Transformer で処理して，文全体を表すベクトルに変換する（注2）．

それぞれのエンコーダは，最終的に同じ次元数の特徴ベクトル（embedding）を出力するように設計されており，この特徴ベクトルの空間では「対応する画像と文」が互いに近くなるように学習が進められる．

<br>
<hr width="50%" align="left">
<span style="font-size: 75%">
[Radford et al., 2021] Radford, A., Kim, J. W., Hallacy, C., et al. (2021). Learning Transferable Visual Models From Natural Language Supervision. arXiv preprint <a href="https://arxiv.org/abs/2103.00020">arXiv:2103.00020</a>.</br>
※注1:
ResNet は，畳み込みニューラルネットの一種である．ViTは，Transformer と呼ばれるニューラルネットモデルの一種であり，その詳細についてはここでは省略する．Transformer は近年注目されているモデルで，特に自然言語処理の分野ではデファクトスタンダードとして広く用いられている．
注2も参照．</br>
※注2: この辺りのことは，「自然言語処理特論I/II」で学べるだろう．
</span>

対照学習とは，特徴ベクトルのペアを考え，対応するペア（例：ある画像の特徴と，それに対応するテキストの特徴）を「正例」，対応しないペアを「負例」として，正例の類似度が大きく，負例の類似度が小さくなるような損失関数にもとづいて学習を行う手法である．
このような学習によって，CLIP は「画像に最もふさわしい説明文を選ぶ」，あるいは「文に合致する画像を選ぶ」といったタスクにおいて非常に高い性能を示すことができる．

後述する Stable Diffusion においては，この CLIP の Text Encoder のみを取り出して使用し，拡散モデルに自然言語の条件を入力することで，テキストに従った画像生成を実現している．


---
### Stable Diffusion

Stable Diffusion は，2022年に公開されて以来，画像生成分野で広く使われている代表的な生成モデルのひとつである（注）．画像とその内容を表すテキスト情報をペアにしたデータで生成モデルを学習させ，テキストによる指示（プロンプト）に従う画像を生成させることができる．

このモデルは，VAE，拡散モデル，そして CLIP の Text Encoder という3つのニューラルネットを組み合わせた構成をとっている．
Stable Diffusion に用いられる拡散モデルは，テキスト $\pmb{y}$ が与えられたときの画像 $\pmb{x}$ の条件付き分布 $p(\pmb{x}|\pmb{y})$ をモデル化する，「条件付き」拡散モデルである．

<br>
<hr width="50%" align="left">
<span style="font-size: 75%">
※注: Stable Diffusion は，生成モデルの名前であるとともに，Stability AI社が提供する生成AIサービスの名前にもなっている．Stable Diffusion の元となったモデルを提案した論文はこちら:</br>
[Rombach et al., 2022] Rombach, R., Blattmann, A., Lorenz, D., Esser, P., & Ommer, B. (2022). High-resolution image synthesis with latent diffusion models. arXiv preprint <a href="https://arxiv.org/abs/2112.10752">arXiv:2112.10752</a>.
</span>


Stable Diffusion モデルの学習は，次のような手続きとなる：

1. 大量の画像で事前学習された VAE を用意する
2. 画像とテキストの対応関係を学習した CLIP を用意する
3. 入力画像を VAE のエンコーダで潜在変数 $ \pmb{x} $ に変換し，
   対応するテキストを CLIP でベクトル $\pmb{y}$に変換する．
   このときのペア $(\pmb{x}, \pmb{y})$ に対して条件付き分布 $ p(\pmb{x} | \pmb{y}) $ をモデル化する拡散モデルを学習する．

学習後のモデルを用いて画像を生成する手続きは次のようになる：

1. 指定されたテキストプロンプトを CLIP によりベクトル $ \pmb{y} $ に変換する
2. $\pmb{y}$ を条件として，拡散モデルによって潜在変数 $\pmb{x} $ を生成する
3. 生成された $\pmb{x}$ を VAE のデコーダに入力して画像を復元する

---
### 実験

実際の Stable Diffusion 動かしてみよう．現在，Stable Diffusion は，
[Stability AI](https://ja.stability.ai/)が開発・提供している．
ウェブ上でも試用できる（ https://stablediffusionweb.com/ )が，学習済みモデルが公開されているので，それを入手して自分のPCで動かすこともできる．ここでは， Colab 上で動かしてみることにする．

事前学習済みモデルをダウンロードする．巨大なモデルでパラメータ数が膨大なので，少し時間がかかる．ここで入手しているのは，SDXL 1.0 というモデル．[Stability AI によるアナウンス](https://stability.ai/news/stable-diffusion-sdxl-1-announcement) によれば，パラメータ数は約35億（注）とのこと．

<br>
<hr width="50%" align="left">
<span style="font-size: 75%">
※注: これは，ここで使用している base model のパラメータ数．追加で，出力を改善する refinement model も使うことができる．そちらもあわせると，総パラメータ数は約
 66 億となる．
</span>


In [ ]:
from diffusers import AutoPipelineForText2Image
import torch

# GPU を使う設定にする
assert torch.cuda.is_available(), 'この実験はGPUが使える環境でないとできません'

# モデルとそのパラメータの入手
pipeline = AutoPipelineForText2Image.from_pretrained('stabilityai/stable-diffusion-xl-base-1.0', torch_dtype=torch.float16, variant='fp16')
pipeline = pipeline.to('cuda')

以下の `prompt` に記したテキスト（プロンプト）をもとに画像を生成する．ランダム性があるので，同じプロンプトでも実行の度に生成結果は変わる．何度か実行してみるとよい．

In [ ]:
prompt = 'a photo of an astronaut riding a horse on mars'
image = pipeline(prompt).images[0]
image

引数でいろいろ指定できる．詳しいことが知りたいひとは，いろいろ調べてみてね．


In [ ]:
prompt = 'high quality, beautiful beach, sunset, sun, palm tree'
neg_prompt = None
image = pipeline(prompt, negative_prompt=neg_prompt, guidance_scale=7.5).images[0]
image

In [ ]:
prompt = 'a high-resolution photorealistic photo of a man facing his laptop computer with a cat sleeping on the keyboard'
neg_prompt = 'painting, drawing, digital art, cartoon, sketch'
image = pipeline(prompt, negative_prompt=neg_prompt, guidance_scale=7.5).images[0]
image

In [ ]:
prompt = 'a figure of mount Fuji erupting with flash of lightning, plume and lava'
neg_prompt = None
image = pipeline(prompt, negative_prompt=neg_prompt, guidance_scale=15.0).images[0]
image

自分で適当なプロンプトを指定していろいろ試してみよう．
https://stablediffusionweb.com/ja の画像をクリックすると，どんなプロンプトで生成されたのかを表示させることができる．そちらを参考にするのもよい．

面白いのができたら takataka に見せていただけると喜びます．

解説はしないが，画像生成モデルの応用場面では，ファインチューニングによって画風・スタイルを調整するようなことも行われる．

参考までに，このモデルに含まれる VAE エンコーダの構造を表示させるコードを示しておく．入力画像の shape が `[3, 1024, 1024]` のとき，VAE エンコーダの出力の shape は `[8, 128, 128]` である．これは，VAEエンコーダが画像を `[4, 128, 128]` という shape の潜在変数に符号化していることを示す．これは，SDXL 1.0 の拡散モデルの潜在変数の shape でもある．

In [ ]:
!pip install torchinfo
from torchinfo import summary

VAE_Encoder = pipeline.vae.encoder.to(dtype=torch.float32).to('cuda')
summary(VAE_Encoder, input_size=((1, 3, 1024, 1024)))